In [3]:
def read_vrp(filename):
    coords = {}
    demands = {}
    capacity = None
    depot = None

    with open(filename, "r") as f:
        section = None

        for line in f:
            line = line.strip()

            if "CAPACITY" in line:
                capacity = Integer(line.split(":")[1])

            elif line == "NODE_COORD_SECTION":
                section = "coords"
                continue

            elif line == "DEMAND_SECTION":
                section = "demands"
                continue

            elif line == "DEPOT_SECTION":
                section = "depot"
                continue

            elif line == "EOF":
                break

            if section == "coords":
                parts = line.split()
                node = Integer(parts[0])
                x = RealNumber(parts[1])
                y = RealNumber(parts[2])
                coords[node] = (x, y)

            elif section == "demands":
                parts = line.split()
                node = Integer(parts[0])
                demand = Integer(parts[1])
                demands[node] = demand

            elif section == "depot":
                if line != "-1":
                    depot = Integer(line)

    return coords, demands, capacity, depot

In [4]:
def compute_distance_matrix(coords):
    dist={}
    
    for i in coords:
        for j in coords:
            x1, y1 = coords[i]
            x2, y2 = coords[j]
            dist[(i, j)] = sqrt((x1 - x2)^2 + (y1 - y2)^2)
    
    return dist

In [5]:
def route_cost(route, dist, depot):
    if not route:
        return 0
    
    total = dist[(depot, route[0])]
    
    for i in range(len(route) - 1):
        total += dist[(route[i], route[i+1])]
        
    total += dist[(route[-1], depot)]
    
    return total

def total_cost(solution, dist, depot):
    return sum(route_cost(route, dist, depot) for route in solution)

In [6]:
def is_feasible(solution, demands, capacity):
    for route in solution:
        load = sum(demands[node] for node in route)
        if load > capacity:
            return False
    return True

In [49]:
def initial_solution(coords, demands, capacity, depot):
    customers = [i for i in coords if i != depot]
    random.shuffle(customers)
    solution = []
    current_route = []
    current_load = 0
    
    for node in customers:
        demand = demands[node]
        
        if current_load + demand <= capacity:
            current_route.append(node)
            current_load += demand
        else:
            solution.append(current_route)
            current_route = [node]
            current_load = demand
        
    if current_route:
        solution.append(current_route)
        
    return solution

In [8]:
# Neighbourhood operators N1 within route - swap
import random

def N1(solution):
    new_solution = [route[:] for route in solution]
    
    route = random.choice(new_solution)
    
    if len(route) < 2:
        return new_solution
    
    i, j = random.sample(range(len(route)), 2)
    route[i], route[j] = route[j], route[i]
    
    return new_solution

In [9]:
# N2 between route - move

def N2(solution, demands, capacity):
    new_solution = [route[:] for route in solution]
    
    if len(new_solution) < 2:
        new_solution.pop(r1)
    
    r1, r2 = random.sample(range(len(new_solution)), 2)
    
    if not new_solution[r1]:
        return new_solution
    
    i = random.randrange(len(new_solution[r1]))
    node = new_solution[r1].pop(i)
    
    new_solution[r2].append(node)
    
    # check feasibility
    if is_feasible(new_solution, demands, capacity):
        return new_solution
    else:
        return solution # reject move

In [32]:
from math import exp
import copy
import random

def simulated_annealing(initial, dist, demands, capacity, depot,
                        max_iter=5000, T=5000, alpha=0.999,
                        mode="N1"):

    current = copy.deepcopy(initial)
    best = copy.deepcopy(initial)

    for _ in range(max_iter):

        # choose neighbourhood
        if mode == "N1":
            neighbour = N1(current)
        elif mode == "N2":
            neighbour = N2(current, demands, capacity)
        else:  # N1N2
            if random.random() < 0.5:
                neighbour = N1(current)
            else:
                neighbour = N2(current, demands, capacity)

        if not is_feasible(neighbour, demands, capacity):
            neighbour = current

        delta = total_cost(neighbour, dist, depot) - total_cost(current, dist, depot)

        if delta < 0 or random.random() < exp(-delta / T):
            current = neighbour

            if total_cost(current, dist, depot) < total_cost(best, dist, depot):
                best = current

        T *= alpha

    return best

In [56]:
def run_multiple(mode, runs=3):
    best_cost = float("inf")
    
    for _ in range(runs):
        initial = initial_solution(coords, demands, capacity, depot)
        sol = simulated_annealing(initial, dist, demands, capacity, depot, mode=mode)
        cost = total_cost(sol, dist, depot)
        
        if cost < best_cost:
            best_cost = cost
    
    return best_cost

In [92]:
coords, demands, capacity, depot = read_vrp("/mnt/d/cvrp-instances/eil22.vrp")
dist = compute_distance_matrix(coords)

initial = initial_solution(coords, demands, capacity, depot)

print("Initial cost:", total_cost(initial, dist, depot))

Initial cost: 775.905584448413


In [93]:
print("N1:", run_multiple("N1"))
print("N2:", run_multiple("N2"))
print("N1N2:", run_multiple("N1N2"))

N1: 665.629522678725
N2: 613.543217684939
N1N2: 546.137149125690


In [34]:
#sol_N1 = simulated_annealing(initial, dist, demands, capacity, depot, mode="N1")
#print("N1 cost:", total_cost(sol_N1, dist, depot))

N1 cost: 519.537310535895


In [35]:
#sol_N2 = simulated_annealing(initial, dist, demands, capacity, depot, mode="N2")
#print("N2 cost:", total_cost(sol_N2, dist, depot))

N2 cost: 558.311130017663


In [36]:
#sol_N1N2 = simulated_annealing(initial, dist, demands, capacity, depot, mode="N1N2")
#print("N1N2 cost:", total_cost(sol_N1N2, dist, depot))

N1N2 cost: 560.890561015908


In [94]:
coords, demands, capacity, depot = read_vrp("/mnt/d/cvrp-instances/eil23.vrp")
dist = compute_distance_matrix(coords)

initial = initial_solution(coords, demands, capacity, depot)

print("Initial cost:", total_cost(initial, dist, depot))

Initial cost: 1322.30505806115


In [38]:
#sol_N1 = simulated_annealing(initial, dist, demands, capacity, depot, mode="N1")
#print("N1 cost:", total_cost(sol_N1, dist, depot))

N1 cost: 870.722014935359


In [39]:
#sol_N2 = simulated_annealing(initial, dist, demands, capacity, depot, mode="N2")
#print("N2 cost:", total_cost(sol_N2, dist, depot))

N2 cost: 990.128594873583


In [40]:
#sol_N1N2 = simulated_annealing(initial, dist, demands, capacity, depot, mode="N1N2")
#print("N1N2 cost:", total_cost(sol_N1N2, dist, depot))

N1N2 cost: 954.507523998705


In [95]:
print("N1:", run_multiple("N1"))
print("N2:", run_multiple("N2"))
print("N1N2:", run_multiple("N1N2"))

N1: 889.648375770305
N2: 923.873472637009
N1N2: 845.381503331866


In [83]:
coords, demands, capacity, depot = read_vrp("/mnt/d/cvrp-instances/eil30.vrp")
dist = compute_distance_matrix(coords)

initial = initial_solution(coords, demands, capacity, depot)

print("Initial cost:", total_cost(initial, dist, depot))

Initial cost: 1826.41735147269


In [42]:
#sol_N1 = simulated_annealing(initial, dist, demands, capacity, depot, mode="N1")
#print("N1 cost:", total_cost(sol_N1, dist, depot))

N1 cost: 942.773428470737


In [43]:
#sol_N2 = simulated_annealing(initial, dist, demands, capacity, depot, mode="N2")
#print("N2 cost:", total_cost(sol_N2, dist, depot))

N2 cost: 930.430595590889


In [44]:
#sol_N1N2 = simulated_annealing(initial, dist, demands, capacity, depot, mode="N1N2")
#print("N1N2 cost:", total_cost(sol_N1N2, dist, depot))

N1N2 cost: 964.039102115926


In [84]:
print("N1:", run_multiple("N1"))
print("N2:", run_multiple("N2"))
print("N1N2:", run_multiple("N1N2"))

N1: 1039.08787203758
N2: 977.955983135473
N1N2: 968.837512160072


In [98]:
coords, demands, capacity, depot = read_vrp("/mnt/d/cvrp-instances/eil33.vrp")
dist = compute_distance_matrix(coords)

initial = initial_solution(coords, demands, capacity, depot)

print("Initial cost:", total_cost(initial, dist, depot))

Initial cost: 1639.86453005990


In [53]:
#sol_N1 = simulated_annealing(initial, dist, demands, capacity, depot, mode="N1")
#print("N1 cost:", total_cost(sol_N1, dist, depot))

N1 cost: 1437.02251499669


In [54]:
#sol_N2 = simulated_annealing(initial, dist, demands, capacity, depot, mode="N2")
#print("N2 cost:", total_cost(sol_N2, dist, depot))

N2 cost: 1436.03780128100


In [55]:
#sol_N1N2 = simulated_annealing(initial, dist, demands, capacity, depot, mode="N1N2")
#print("N1N2 cost:", total_cost(sol_N1N2, dist, depot))

N1N2 cost: 1331.25212591805


In [99]:
print("N1:", run_multiple("N1"))
print("N2:", run_multiple("N2"))
print("N1N2:", run_multiple("N1N2"))

N1: 1492.85628908910
N2: 1313.18975463599
N1N2: 1226.79961397839
